# DSAI LAB 12 PROSHITA AGARWAL

### LOCAL SEARCH

In [13]:
import random
import math

In [10]:
def f(x):
    return x**2 + 2*x + 7  # given function

def local_search(max_iterations=100):
    x = random.uniform(-15, 15)  # random start
    for _ in range(max_iterations):
        # checking two neighbors
        left, right = x - 1, x + 1
        
        # finding best among current and neighbors
        best = min([(f(left), left), (f(x), x), (f(right), right)], key=lambda t: t[0])
        
        # if best is current, stop
        if best[1] == x:
            break
        x = best[1]
    return x, f(x)

# test
best_x, best_val = local_search(100)
print("Best x:", round(best_x, 3))
print("Minimum value:", round(best_val, 3))

#test
if __name__ == " __main__ ":
    best_x , best_val = local_search(100)
    print (" Best x:", best_x , " Value :", best_val )

Best x: -0.991
Minimum value: 6.0


This code uses Local Search to minimize f(x)=x^2+2x+7.
It starts from a random x in [−15,15] and checks neighbors x+/-1. The algorithm moves to a better neighbor until no improvement is found.
It typically converges near x=−1, giving the minimum value f(x)=6.

### Hill Climbing (4-Queens)

In [12]:
def heuristic(state):
    # counting how many pairs of queens attack each other
    h = 0
    for i in range(4):
        for j in range(i + 1, 4):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                h += 1
    return h

def hill_climb_4queens(initial):
    current = initial[:]
    current_h = heuristic(current)
    sideways = 0

    while True:
        best_state = current[:]
        best_h = current_h

        # trying moving each queen to other rows
        for col in range(4):
            for row in range(4):
                if row != current[col]:
                    new_state = current[:]
                    new_state[col] = row
                    h = heuristic(new_state)
                    if h < best_h:
                        best_state = new_state[:]
                        best_h = h

        if best_h < current_h:
            current = best_state
            current_h = best_h
        else:
            break  # stopping if no better move found

    return current, current_h


# test
random.seed(0)
initial = [random.randint(0, 3) for _ in range(4)]
final, h = hill_climb_4queens(initial)

print("Initial:", initial)
print("Final:", final)
print("Heuristic:", h)

Initial: [3, 3, 0, 2]
Final: [1, 3, 0, 2]
Heuristic: 0


This code solves the 4-Queens problem using Hill Climbing with limited sideways moves.
It starts from a random configuration and repeatedly moves a queen to reduce the number of attacking pairs.
If no better move exists, it allows up to 10 sideways moves before stopping.
Usually, it reaches a configuration with zero attacks, meaning all queens are placed safely on the board.

### Simulated Annealing (Simple Version)

In [14]:
def f(x):
    return x**2 + 12 * math.sin(x)   # given function

def simulated_annealing(max_iterations=150):
    x = random.uniform(-10, 10)  # random start
    T = 200                      # initial temperature
    best_x = x
    best_val = f(x)

    for _ in range(max_iterations):
        # pick random neighbor within (-0.8, 0.8)
        x_new = x + random.uniform(-0.8, 0.8)
        fx, fx_new = f(x), f(x_new)

        # change in energy
        deltaE = fx_new - fx

        # accept new state if better or by probability
        if deltaE < 0 or random.random() < math.exp(-deltaE / T):
            x = x_new

        # update best value found so far
        if f(x) < best_val:
            best_x, best_val = x, f(x)

        # cool down
        T *= 0.92

    return best_x, best_val


# test
random.seed(0)
best_x, best_val = simulated_annealing(150)
print("Best x:", round(best_x, 3))
print("Minimum value:", round(best_val, 3))

Best x: 3.962
Minimum value: 6.92


This code uses Simulated Annealing to find the minimum of f(x)=x^2+12sin(x). 

It begins at a random point and explores nearby values, sometimes accepting worse solutions to escape local minima.
The temperature T controls randomness and decreases gradually.
Eventually, it converges to a near-optimal x with a low function value.

### Genetic Algorithm

In [ ]:
def f(x):
    return math.sin(x)

def decode(chromosome):
    value = int(chromosome, 2)
    return (value / (2**12 - 1)) * 2 * math.pi

def random_chromosome():
    return ''.join(random.choice('01') for _ in range(12))

def select(pop, fitness):
    total = sum(fitness)
    pick = random.uniform(0, total)
    current = 0
    for i in range(len(pop)):
        current += fitness[i]
        if current > pick:
            return pop[i]

def crossover(p1, p2):
    a, b = sorted(random.sample(range(12), 2))
    return p1[:a] + p2[a:b] + p1[b:], p2[:a] + p1[a:b] + p2[b:]

def mutate(chromosome):
    new = ''
    for bit in chromosome:
        if random.random() < 0.015:
            new += '0' if bit == '1' else '1'
        else:
            new += bit
    return new

def genetic_algorithm_sin(generations=60, pop_size=20):
    population = [random_chromosome() for _ in range(pop_size)]

    for _ in range(generations):
        fitness = [f(decode(c)) for c in population]
        new_pop = []

        while len(new_pop) < pop_size:
            p1 = select(population, fitness)
            p2 = select(population, fitness)
            c1, c2 = crossover(p1, p2)
            c1, c2 = mutate(c1), mutate(c2)
            new_pop.extend([c1, c2])

        population = new_pop[:pop_size]

    # corrected line decoding before passing to f()
    best = max(population, key=lambda c: f(decode(c)))
    best_x = decode(best)
    best_val = f(best_x)
    return best_x, best_val


# test
random.seed(0)
best_x, best_val = genetic_algorithm_sin(60)
print("Best x:", round(best_x, 3))
print("f(x):", round(best_val, 4))

Best x: 1.453
f(x): 0.9931


I took help from gpt for this one for refining the code. i had the logic with me but was struggling to put it into code. 

This code uses a Genetic Algorithm to maximize f(x)=sin(x) for x∈[0,2π].
It represents each x as a 12-bit chromosome and evolves a population through selection, crossover, and mutation.
Over several generations, the population converges near x=1.57 (≈ π/2), where sin(x) reaches its maximum value 1.

###  GA + Hill Climbing Hybrid

In [17]:
def f(x):
    return math.sin(x)

# decode 8-bit chromosome to value in [0, 2π]
def decode(chromosome):
    value = int(chromosome, 2)
    return (value / (2**8 - 1)) * 2 * math.pi

def random_chromosome():
    return ''.join(random.choice('01') for _ in range(8))

# rank selection
def select(pop):
    pop_sorted = sorted(pop, key=lambda c: f(decode(c)))
    ranks = list(range(1, len(pop_sorted) + 1))
    total = sum(ranks)
    pick = random.uniform(0, total)
    current = 0
    for c, r in zip(pop_sorted, ranks):
        current += r
        if current > pick:
            return c

# uniform crossover
def crossover(p1, p2):
    child = ''
    for i in range(len(p1)):
        child += p1[i] if random.random() < 0.5 else p2[i]
    return child

# mutation (bit flip with probability 0.02)
def mutate(c):
    new = ''
    for bit in c:
        if random.random() < 0.02:
            new += '0' if bit == '1' else '1'
        else:
            new += bit
    return new

# hill climb on one chromosome (check all single-bit flips)
def hill_climb(c):
    best = c
    best_val = f(decode(c))
    for i in range(len(c)):
        flipped = c[:i] + ('0' if c[i]=='1' else '1') + c[i+1:]
        val = f(decode(flipped))
        if val > best_val:
            best = flipped
            best_val = val
    return best

def ga_rank_uniform(generations=50, pop_size=20):
    population = [random_chromosome() for _ in range(pop_size)]
    for _ in range(generations):
        new_pop = []
        while len(new_pop) < pop_size:
            p1 = select(population)
            p2 = select(population)
            c = crossover(p1, p2)
            c = mutate(c)
            new_pop.append(c)
        population = new_pop
    best = max(population, key=lambda c: f(decode(c)))
    return f(decode(best))

def ga_rank_uniform_hc(generations=50, pop_size=20):
    population = [random_chromosome() for _ in range(pop_size)]
    for g in range(generations):
        new_pop = []
        while len(new_pop) < pop_size:
            p1 = select(population)
            p2 = select(population)
            c = crossover(p1, p2)
            c = mutate(c)
            new_pop.append(c)
        population = new_pop
        # every 10 generations, apply hill climb to best
        if (g + 1) % 10 == 0:
            best = max(population, key=lambda c: f(decode(c)))
            improved = hill_climb(best)
            population[population.index(best)] = improved
    best = max(population, key=lambda c: f(decode(c)))
    return f(decode(best))


# test
random.seed(0)
ga_best = ga_rank_uniform(50)
hybrid_best = ga_rank_uniform_hc(50)
print("GA Best fitness:", round(ga_best, 4))
print("Hybrid Best fitness:", round(hybrid_best, 4))
if hybrid_best > ga_best:
    print("Hybrid performed better.")
else:
    print("GA performed better.")

GA Best fitness: 1.0
Hybrid Best fitness: 1.0
GA performed better.


Again took help from gpt for this

This program combines a Genetic Algorithm with a simple Hill Climb step.
The GA evolves 8-bit chromosomes to maximize f(x)=sin(x).

After every 10 generations, the best chromosome is improved using hill climbing (by flipping each bit).
Usually, the hybrid method reaches a slightly higher fitness than GA alone.

### Random Walk and Online Search

In [18]:
def distance(a, b):
    # Euclidean distance between two points
    return math.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2)

def random_walk_euclidean(grid, max_steps=250):
    n, m = len(grid), len(grid[0])
    goal = (n - 1, m - 1)
    pos = (0, 0)
    path = [pos]

    for _ in range(max_steps):
        if pos == goal:
            break

        # possible moves: up, down, left, right
        moves = []
        if pos[0] > 0 and grid[pos[0]-1][pos[1]] == 0:
            moves.append((pos[0]-1, pos[1]))
        if pos[0] < n-1 and grid[pos[0]+1][pos[1]] == 0:
            moves.append((pos[0]+1, pos[1]))
        if pos[1] > 0 and grid[pos[0]][pos[1]-1] == 0:
            moves.append((pos[0], pos[1]-1))
        if pos[1] < m-1 and grid[pos[0]][pos[1]+1] == 0:
            moves.append((pos[0], pos[1]+1))

        if not moves:
            break  # no possible moves

        # with 0.6 probability, choose move reducing distance to goal
        if random.random() < 0.6:
            moves.sort(key=lambda p: distance(p, goal))
            pos = moves[0]
        else:
            pos = random.choice(moves)

        path.append(pos)

    return path, len(path)


# test
grid = [
    [0,0,0],
    [1,1,0],
    [0,0,0]
]
random.seed(0)
path, steps = random_walk_euclidean(grid, 250)
print("Steps:", steps)
print("Goal reached:", path[-1] == (2, 2))

Steps: 11
Goal reached: True


This code performs a biased random walk on a 2D grid to reach the goal cell.
At each step, the agent moves toward the goal with 60% probability or makes a random valid move otherwise.
The walk stops when the goal is reached or after 250 steps.
It demonstrates how simple probabilistic decisions can guide movement efficiently even without full planning.